# Soi chart gây hiểu nhầm: viết một "linter" cho matplotlib

Một chart sai hiếm khi do cố ý: trục y tự co về dải dữ liệu, `twinx()` một dòng là có hai trục, pie
mặc định chia mọi nhóm. DA senior bắt các lỗi này trong review, ở chart của mình và của đồng nghiệp.
Bài này biến năm lỗi hay gặp thành luật kiểm tự động trên đối tượng `Figure` của matplotlib, rồi áp
lên bốn chart "xấu" vẽ từ dữ liệu thật (Olist và Bike Sharing).

| Mã lỗi | Khi nào báo |
| --- | --- |
| `truncated_bar_axis` | Axes có bar mà trục giá trị (y với bar đứng, x với bar ngang) không bắt đầu từ 0 hoặc thấp hơn |
| `dual_axis` | Figure có hai Axes chồng đúng một vị trí (kết quả của `twinx()`/`twiny()`) |
| `pie_too_many_slices` | Pie có hơn 6 lát |
| `unsorted_bars` | Bar trên trục **danh mục** (nhãn chữ) mà chiều cao không đơn điệu theo thứ tự trên trục |
| `missing_axis_label` | Một trục đang hiện (có vạch chia) nhưng nhãn trục rỗng; Axes vẽ pie được miễn |

Đây là linter: nó chỉ ra **chỗ nghi ngờ**, người vẽ quyết định. Thứ trong tuần hay tháng là danh mục có
thứ tự tự nhiên, linter sẽ báo `unsorted_bars` và bạn có quyền bỏ qua. Ngược lại có những chart qua
hết luật mà vẫn lừa người xem (kỳ dở dang, tổng vs trung bình); phần trắc nghiệm hỏi về chúng.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`; hàm không được sửa dữ liệu đầu vào.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) đọc dữ liệu thật và vẽ; khi chấm chúng bị bỏ qua.
- Chấm: `uv run pytest tracks/data-analyst/modules/08-visualization-chart-reading/exercises/intermediate/01-misleading-chart-audit -q`

In [ ]:
# explore: tìm gốc repo, tải Bike Sharing nếu chưa có, đọc Olist thật (tải bằng: uv run python data/download_olist.py)
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure
import pandas as pd

BIKE = ensure("bike-sharing-uci")
day = pd.read_csv(BIKE / "day.csv", parse_dates=["dteday"])

RAW = ROOT / "data" / "raw"
if not (RAW / "olist_orders_dataset.csv").exists():
    raise FileNotFoundError("Chưa có Olist thật. Chạy: uv run python data/download_olist.py")
payments = pd.read_csv(RAW / "olist_order_payments_dataset.csv")
reviews = pd.read_csv(RAW / "olist_order_reviews_dataset.csv")
customers = pd.read_csv(RAW / "olist_customers_dataset.csv")
orders = pd.read_csv(RAW / "olist_orders_dataset.csv")
print(len(day), "ngày Bike Sharing;", len(orders), "đơn,", len(payments), "dòng thanh toán,", len(reviews), "review Olist")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.axes import Axes
from matplotlib.category import StrCategoryConverter
from matplotlib.container import BarContainer
from matplotlib.figure import Figure
from matplotlib.patches import Wedge

ISSUES = {"truncated_bar_axis", "dual_axis", "pie_too_many_slices", "unsorted_bars", "missing_axis_label"}
MAX_PIE_SLICES = 6

## 1. Luật kiểm trên một `Figure`

`audit_chart(fig) -> set[str]` trả tập mã lỗi (tập con của `ISSUES`, rỗng nếu sạch). Những gì bạn cần
biết về cấu trúc bên trong một Figure:

- `fig.axes`: mọi Axes, kể cả Axes sinh bởi `twinx()`. `ax.get_position().bounds` là khung
  `(x0, y0, rộng, cao)` trên figure; hai Axes twin có khung trùng nhau, hai panel cạnh nhau thì không.
- `ax.containers`: mỗi lần gọi `ax.bar`/`ax.barh`/`ax.hist` thêm một `BarContainer`; thuộc tính
  `orientation` là `"vertical"` hoặc `"horizontal"`, mỗi phần tử là một `Rectangle`
  (`get_x()`, `get_y()`, `get_width()`, `get_height()`).
- `ax.patches`: mọi patch; lát pie là `matplotlib.patches.Wedge`.
- `ax.get_ylim()`, `ax.get_xlim()`: giới hạn trục (có thể đảo ngược sau `invert_yaxis()`: lấy `min`).
- Trục danh mục: `ax.bar(["SP", "RJ"], ...)` gắn converter `StrCategoryConverter` vào trục
  (`axis.get_converter()`). Pandas `Series.plot.bar()` thì vẽ ở vị trí `0..n-1` rồi ghi nhãn chữ lên
  vạch chia: không có converter nhưng nhãn vạch chia (`axis.get_ticklabels()`) không phải số. Trục ngày
  (`DatetimeIndex`) có converter riêng và **không** phải danh mục.
- `axis.get_visible()`: trục x của Axes twin bị ẩn; `len(axis.get_ticklocs()) == 0` khi bạn đã
  `set_yticks([])` (nhãn trực tiếp thay cho trục): khi đó không cần nhãn trục.

Bar "đã sắp" nghĩa là chiều cao đơn điệu (không tăng hoặc không giảm) theo vị trí trên trục danh mục;
`barh` tăng dần từ dưới lên là đã sắp (cột dài nhất trên cùng). Chỉ xét container có từ 3 bar.

In [ ]:
def _is_category_axis(axis) -> bool:
    """True nếu trục mang nhãn danh mục (chuỗi), không phải số hay ngày."""
    # TODO: converter là StrCategoryConverter → True; converter khác (ngày) → False
    # TODO: không có converter: True nếu có nhãn vạch chia không đọc được thành số (bar của pandas)
    raise NotImplementedError


def audit_chart(fig: Figure) -> set[str]:
    """Tập mã lỗi trong ISSUES tìm thấy trên figure."""
    # TODO: dual_axis: hai Axes có cùng get_position().bounds
    # TODO: với mỗi Axes: đếm Wedge (pie), duyệt BarContainer (trục cắt, thứ tự), kiểm nhãn trục
    raise NotImplementedError

## 2. Bar trung thực

`honest_bar(ax, s) -> Axes`: vẽ `s` thành bar đứng trên `ax`, **sắp giảm dần**, trục y từ 0, nhãn trục x
là `s.index.name`, nhãn trục y là `s.name` (thiếu một trong hai thì `raise ValueError`: một chart không
có đơn vị là chart chưa xong). Trả lại chính `ax`. `audit_chart(honest_bar(ax, s).figure)` phải là tập
rỗng.

In [ ]:
def honest_bar(ax: Axes, s: pd.Series) -> Axes:
    """Bar sắp giảm dần, y từ 0, nhãn trục lấy từ s.index.name và s.name."""
    # TODO: ValueError nếu thiếu tên; sắp giảm dần; ax.bar với nhãn chuỗi; ylim từ 0; nhãn trục
    raise NotImplementedError

## 3. Bốn chart "xấu" từ dữ liệu thật

Mỗi chart dưới đây là kiểu chart bạn sẽ gặp trong slide thật. Trước khi chạy `audit_chart`, tự đọc
từng chart theo bước 1 của quy trình (trục, đơn vị, thang) và ghi ra lỗi bạn thấy. Sau đó so với linter.

In [ ]:
# explore: chart 1 - điểm review trung bình theo phương thức thanh toán (Olist)
first_pay = payments[payments.payment_sequential == 1][["order_id", "payment_type"]]
score = (reviews.groupby("order_id").review_score.mean().rename("review_score").reset_index()
         .merge(first_pay, on="order_id"))
by_pay = (score[score.payment_type != "not_defined"].groupby("payment_type").review_score
          .agg(["mean", "size"]).sort_values("mean", ascending=False))
print(by_pay.round(3))

fig1, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(by_pay.index, by_pay["mean"], color="#4c78a8")
ax.set_ylim(by_pay["mean"].min() - 0.02, by_pay["mean"].max() + 0.02)
ax.set_xlabel("phương thức thanh toán")
ax.set_ylabel("điểm review TB (1-5)")
ax.set_title("Khách trả bằng debit card hài lòng hơn hẳn?", loc="left")
plt.show()

In [ ]:
# explore: chart 2 - khách vãng lai và nhiệt độ theo tháng (Bike Sharing)
monthly = day.set_index("dteday").resample("MS").agg({"casual": "sum", "temp": "mean"})
fig2, ax = plt.subplots(figsize=(8, 3.5))
ax.bar(monthly.index, monthly.casual / 1000, width=20, color="#9aa5b1")
ax.set_xlabel("tháng")
ax.set_ylabel("lượt khách vãng lai (nghìn)")
ax2 = ax.twinx()
ax2.plot(monthly.index, monthly.temp * 41, color="#f58518", marker="o")   # temp chuẩn hoá theo max 41°C
ax2.set_ylabel("nhiệt độ TB (°C)")
ax2.set_ylim(0, 35)
ax.set_title("Nhiệt độ quyết định lượng khách vãng lai", loc="left")
plt.show()
print(f"tương quan theo tháng: {monthly.casual.corr(monthly.temp):.2f}")

In [ ]:
# explore: chart 3 - doanh thu theo bang của khách (Olist)
state_rev = (payments.merge(orders[["order_id", "customer_id"]], on="order_id")
             .merge(customers[["customer_id", "customer_state"]], on="customer_id")
             .groupby("customer_state").payment_value.sum().sort_values(ascending=False))
fig3, ax = plt.subplots(figsize=(6, 6))
ax.pie(state_rev, labels=state_rev.index, textprops={"fontsize": 7})
ax.set_title(f"Doanh thu theo bang ({len(state_rev)} bang)", loc="left")
plt.show()
print((state_rev / state_rev.sum()).head(5).round(3).to_dict())

In [ ]:
# explore: chart 4 - số giao dịch theo phương thức thanh toán (Olist)
pay_count = payments.payment_type.value_counts().sort_index()      # sắp theo bảng chữ cái
fig4, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(pay_count.index, pay_count.values, color="#4c78a8")
ax.set_title("Giao dịch theo phương thức thanh toán", loc="left")
plt.show()

In [ ]:
# explore: linter nói gì về bốn chart? So với danh sách bạn tự ghi.
for name, fig in [("1 review theo thanh toán", fig1), ("2 vãng lai vs nhiệt độ", fig2),
                  ("3 pie doanh thu theo bang", fig3), ("4 giao dịch theo thanh toán", fig4)]:
    print(f"chart {name}: {sorted(audit_chart(fig))}")

## 4. Sửa, rồi đọc lại

Vẽ lại chart 1 và chart 4 bằng `honest_bar`. Với chart 1, nhìn bản trung thực: khác biệt "hẳn" giữa các
phương thức còn lại bao nhiêu điểm trên thang 1-5? Nếu khác biệt nhỏ mà quan trọng, cách trung thực để
phóng to là vẽ **chênh lệch** so với trung bình (hoặc dot plot) chứ không cắt trục bar.

Chart 2 không sửa được bằng một hàm: hai thang đo tự chọn. Thử vẽ hai panel xếp chồng chung trục x
(`plt.subplots(2, 1, sharex=True)`) rồi tự hỏi headline "nhiệt độ quyết định" còn đứng vững không:
tháng nào nóng nhất, tháng nào đông khách vãng lai nhất? Chart 3: bar ngang top 8 bang + "Khác".

Xong thì mở app, làm phần **trắc nghiệm** của bài.

In [ ]:
# explore: bản trung thực của chart 1 và chart 4
fig, (a, b) = plt.subplots(1, 2, figsize=(12, 3.5))
honest_bar(a, by_pay["mean"].rename("điểm review TB (1-5)").rename_axis("phương thức thanh toán"))
a.set_title(f"Chênh lệch lớn nhất chỉ {by_pay['mean'].max() - by_pay['mean'].min():.2f} điểm", loc="left")
honest_bar(b, (pay_count / 1000).rename("giao dịch (nghìn)").rename_axis("phương thức thanh toán"))
b.set_title(f"credit_card chiếm {pay_count.max() / pay_count.sum():.0%} giao dịch", loc="left")
fig.tight_layout()
plt.show()
print("linter:", sorted(audit_chart(fig)))

In [ ]:
# explore: chart 2 vẽ lại thành hai panel chung trục x
fig, (top, bottom) = plt.subplots(2, 1, figsize=(8, 5), sharex=True)
top.plot(monthly.index, monthly.casual / 1000, marker="o", color="#4c78a8")
top.set_ylabel("vãng lai (nghìn)")
top.set_ylim(0, None)
bottom.plot(monthly.index, monthly.temp * 41, marker="o", color="#f58518")
bottom.set_ylabel("nhiệt độ TB (°C)")
bottom.set_xlabel("tháng")
for a in (top, bottom):
    a.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
plt.show()
print("tháng nóng nhất:", monthly.temp.idxmax().strftime("%Y-%m"), "| tháng đông vãng lai nhất:",
      monthly.casual.idxmax().strftime("%Y-%m"))